# L9 In-Class Activity: Take Apart Two Apps from the Command Line

### E190BO — Fall 2026

In the video, you watched Hyperschedule get taken apart one layer at a time. Today you will do it yourself using this notebook.

- **Toolbox (about 5 min).** Short demos of every tool you'll need: `curl`, pipes, headers, status codes, and `jq`. Run them and read what they print.
- **Part 1 — Hyperschedule (about 22 min).** The same API from the video. **You write every command**: each question's code cell starts empty. Check your answers against the **checkpoint key**.
- **Part 2 — Wordle (about 25 min).** A new app, and this time no one is giving you the URLs. Find them with your browser's devtools, then take the app apart with the same tools. It ends with the same diagram as the video, filled in for Wordle.

**Work in teams of 2–3.** Everyone runs this notebook in their own codespace and types every command. Don't move on until everyone on your team has the output. Your team shares one paper worksheet.

Aim to finish Part 1 by **10:05**. If you're not done by then, move on to Part 2 anyway; that's where the sleuthing is.

## How this notebook runs shell commands

Every code cell here starts with **`%%bash`**. That first line tells Jupyter to run the rest of the cell in **bash**, the same shell as your Codespace terminal. Anything that works in these cells works in the terminal too (**Terminal → New Terminal**, then paste everything except the `%%bash` line).

The Toolbox cells below are filled in. After that, each code cell holds only `%%bash` and a comment: replace the comment with your command. (Lines starting with `#` are comments; bash ignores them.) **Keep the `%%bash` line.** Without it, Jupyter reads your command as Python and reports a `SyntaxError`.

Two differences from a terminal:

- **Each cell starts a fresh shell.** A variable you set in one cell (`X=5`) is gone by the next one. The exception is `%env`, which sets a variable for every cell. We use it once, just below.
- You may see `!curl …` in other notebooks. **Don't use it here.** Jupyter rewrites `{…}` and `$name` in `!` lines before bash ever sees them, and that breaks most `jq` commands.

## Setup — run this once

`$B` is shorthand for Hyperschedule's API, just like in the video. `%env` makes it available in every cell below.

In [12]:
%env B=https://banana.hyperschedule.io

env: B=https://banana.hyperschedule.io


---
# Toolbox

Run each demo and read what it prints. The demos use the **course website** and a small **made-up file**, so none of them answers a question in Part 1 or Part 2.

Use this section as a reference whenever you need a pattern.

### `curl` fetches a URL

`curl` prints whatever the server sends back: here, the course home page, which is HTML. `-s` (silent) hides the progress meter.

The `|` (a **pipe**) sends one command's output into the next command instead of to the screen. `head -n 5` keeps only the first five lines.

In [2]:
%%bash
curl -s https://course.aiprototypes.org/ | head -n 5

<!DOCTYPE html>
<html xmlns="http://www.w3.org/1999/xhtml" lang="en" xml:lang="en"><head>

<meta charset="utf-8">
<meta name="generator" content="quarto-1.10.18">


### Save it and measure it

`>` saves the output to a file instead of printing it, and `wc -c` counts the file's bytes.

The last line gets the same number straight from `curl`: `-o /dev/null` throws the body away, and `-w` writes out a fact about the transfer, here its size.

In [3]:
%%bash
curl -s https://course.aiprototypes.org/ > home.html
wc -c home.html
curl -s -o /dev/null -w '%{size_download} bytes\n' https://course.aiprototypes.org/

28208 home.html
28208 bytes


### Headers

Before the body, a server sends **headers**: a status line (`HTTP/2 200`), then `name: value` lines about the response. `-D -` prints the headers, and `-o /dev/null` throws the body away.

These are the headers for the notebook you downloaded at the start of class. Find its type (`content-type`), its size in bytes (`content-length`), how long it may be cached (`cache-control`), and who served it (`server`).

In [6]:
%%bash
curl -sS -D - -o /dev/null https://course.aiprototypes.org/resources/files/eng190bo-fa26-lecture-9-activity-skeleton.ipynb

HTTP/2 200 
date: Wed, 30 Sep 2026 17:04:50 GMT
content-type: application/octet-stream
content-length: 27343
x-content-type-options: nosniff
report-to: {"group":"cf-nel","max_age":604800,"endpoints":[{"url":"https://a.nel.cloudflare.com/report/v4?s=fTqh4V5Ft6RS2QM2YprkfW083fA6gJMeKYFCcsZQEZWvxtBruoOmf3CxALpP7FGAi%2BGBdgFFfmbsolhuzkkLXW1TbytRAW0nNWdYH2wHPmEsC6p5z%2FBM%2FgMA3uEF%2FBWJe%2Bc2NztXpbIHkA%3D%3D"}]}
nel: {"report_to":"cf-nel","success_fraction":0.0,"max_age":604800}
access-control-allow-origin: *
cache-control: public, max-age=0, must-revalidate
etag: "b6151bd04ae006924c09483d3b0fd675"
referrer-policy: strict-origin-when-cross-origin
server: cloudflare
cf-cache-status: DYNAMIC
cf-ray: a434cf1dcbbd4bdc-MXP
alt-svc: h3=":443"; ma=86400



`grep` keeps only the lines that match. `-i` ignores upper and lower case, and `^` means "at the start of a line," so `'^server'` finds the `server:` header and nothing else.

In [7]:
%%bash
curl -sS -D - -o /dev/null https://course.aiprototypes.org/ | grep -i '^server'

server: cloudflare


### Status codes

`-w '%{http_code}'` prints just the status code, as in the first line.

To see the body **and** the code, leave out `-o /dev/null` and use `-w '\n[%{http_code}]\n'`, which puts the code on its own line after the body. The home page is long, so the second line keeps only its last two lines: the end of the HTML, then the code.

In [8]:
%%bash
curl -s -o /dev/null -w '%{http_code}\n' https://course.aiprototypes.org/
curl -s -w '\n[%{http_code}]\n' https://course.aiprototypes.org/ | tail -n 2

200
</body></html>
[200]


### `jq` reads JSON

Most APIs answer in **JSON**, and `jq` is a small language for pulling pieces out of it. To practice, this cell writes a made-up library catalog to `books.json` (everything between `<<'EOF'` and `EOF` goes into the file). Then `jq .` pretty-prints it; `.` means "the whole thing."

In [9]:
%%bash
cat > books.json <<'EOF'
[
  {"title": "A Wizard of Earthsea", "author": {"first": "Ursula", "last": "Le Guin"}, "pages": 183, "copies": 3, "checkedOut": 3},
  {"title": "The Dispossessed", "author": {"first": "Ursula", "last": "Le Guin"}, "pages": 387, "copies": 2, "checkedOut": 1},
  {"title": "Kindred", "author": {"first": "Octavia", "last": "Butler"}, "pages": 264, "copies": 4, "checkedOut": 4},
  {"title": "Exhalation", "author": {"first": "Ted", "last": "Chiang"}, "pages": 350, "copies": 2, "checkedOut": 0}
]
EOF
jq . books.json

[
  {
    "title": "A Wizard of Earthsea",
    "author": {
      "first": "Ursula",
      "last": "Le Guin"
    },
    "pages": 183,
    "copies": 3,
    "checkedOut": 3
  },
  {
    "title": "The Dispossessed",
    "author": {
      "first": "Ursula",
      "last": "Le Guin"
    },
    "pages": 387,
    "copies": 2,
    "checkedOut": 1
  },
  {
    "title": "Kindred",
    "author": {
      "first": "Octavia",
      "last": "Butler"
    },
    "pages": 264,
    "copies": 4,
    "checkedOut": 4
  },
  {
    "title": "Exhalation",
    "author": {
      "first": "Ted",
      "last": "Chiang"
    },
    "pages": 350,
    "copies": 2,
    "checkedOut": 0
  }
]


`length` counts the items in a list. `.[0]` is the first item (counting starts at 0). A dot path walks into it: `.[0].author.last` is the first book's author's last name. Separate paths with commas to print several at once.

In [10]:
%%bash
jq length books.json
jq '.[0]' books.json
jq '.[0].title, .[0].author.last' books.json

4
{
  "title": "A Wizard of Earthsea",
  "author": {
    "first": "Ursula",
    "last": "Le Guin"
  },
  "pages": 183,
  "copies": 3,
  "checkedOut": 3
}
"A Wizard of Earthsea"
"Le Guin"


### Cheat sheet

| Command | Does |
|---|---|
| `curl -s URL` | print the response body |
| `curl -s URL \| jq .` | …as pretty-printed JSON |
| `curl -s URL > file.json` | save the body to a file instead of printing it |
| `jq length file.json` | how many items are in the list? |
| `jq '.[0].title' file.json` | one field of the first item |
| `jq '.[] \| select(.a == "x") \| .title' file.json` | a field of every item that passes a test |
| `jq '[.[] \| select(TEST)] \| length' file.json` | how many items pass a test |
| `jq -c '.[] \| {title, half: (.pages / 2)}' file.json` | pick and compute fields, one line per item |
| `curl -sS -D - -o /dev/null URL` | headers only (the status is on the first line) |
| `curl -s -o /dev/null -w '%{http_code}\n' URL` | just the status code |
| `curl -s -w '\n[%{http_code}]\n' URL` | the body, then the status code |
| `curl -s -o /dev/null -w '%{size_download}\n' URL` | how many bytes came back |
| `… \| grep -i '^word'` | keep only the lines that start with *word* |
| `… \| head -n 5` | keep only the first five lines |
| `date -d '2026-09-30 - 7 days' +%F` | date arithmetic |
| `for x in A B C; do …; done` | run a command once for each value |

---
# Part 1 — Hyperschedule

Same API, same diagram as the video. Every question matches a layer you saw.

## The API server: talk to it

### 1.1 What term is it?

Ask the API for the current term. The route is `/v4/term/current`, so the URL is `$B/v4/term/current`. This is the first request from the video.

In [19]:
%%bash
# write your command here
curl -X GET $B/v4/term/current


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    34    0    34    0     0     26      0 --:--:--  0:00:01 --:--:--    26


{
  "year": 2026,
  "term": "FA"
}

Which term and year does Hyperschedule think it is? What format is the response in?

**Your answer:**
Fall 2026



### 1.2 How many sections?

`$B/v4/sections` returns every section this term, as one list. Count them.

Then **save** the whole list to a file named `sections.json`. It's about 2.5 MB, so from here on read the file with `jq` rather than downloading it over and over. That's faster for you and politer to the server.

In [23]:
%%bash
# write your command here
curl -s GET $B/v4/sections > sections.json

How many sections are offered this term?

**Your answer:** 2185




### 1.3 One section up close

Print the **first** section in `sections.json`, so you can see every field.

In [30]:
%%bash
# write your command here
jq '.[0]' sections.json

{
  "courseAreas": [
    "1A3",
    "2FYA",
    "2RES",
    "4HSS",
    "5HUM",
    "5IUG",
    "5IUL",
    "AFRI"
  ],
  "credits": 1,
  "permCount": 0,
  "seatsTotal": 25,
  "seatsFilled": 23,
  "status": "R",
  "startDate": {
    "year": 2026,
    "month": 8,
    "day": 31
  },
  "endDate": {
    "year": 2026,
    "month": 12,
    "day": 18
  },
  "instructors": [
    {
      "name": "Maryan Soliman"
    }
  ],
  "course": {
    "code": {
      "department": "AFRI",
      "courseNumber": 10,
      "suffix": "A",
      "affiliation": "AF"
    },
    "title": "Intro to Africana Studies",
    "description": "This class will serve as a general introduction to Africana Studies. Africana studies, while still relatively young, has a vibrant history that traces the lives and scholarship of people from African descent. Its complex and latent development in academia follows from the socio-political marginalization of people within the African diaspora. Nevertheless, resilience and perseveranc

Now print **only** three things about that section: its course title, how many seats are filled, and how many there are in total. Find the field names in the output above. The title is inside `course`.

In [36]:
%%bash
# write your command here
jq '.[0].course.title, .[0].seatsFilled, .[0].seatsTotal' sections.json


"Intro to Africana Studies"
23
25


Title, seats filled, seats total:

**Your answer:** 
Intro to Africana Studies,
23,
25

### 1.4 Find this class

How many sections are in the **ENGR** department? The department is at `.course.code.department`.

In [46]:
%%bash
# write your command here
jq '[.[] | select(.course.code.department == "ENGR")] | length' sections.json

55


Now find **ENGR 190BO**, this class, and print its title and seat counts.

Hints: combine tests with `and`. The course number is `.course.code.courseNumber`, and it's stored as a number (`190`, not `"190"`). The `BO` is stored separately, in `.course.code.suffix`. Build a small object to print, as in the cheat sheet.

In [ ]:
%%bash
# write your command here


How many ENGR sections are there? Is there still room in ENGR 190BO?

**Your answer:**



## The network

### 1.5 How much did the browser download?

How many bytes does `$B/v4/sections` send back? The Toolbox shows two ways: `curl`'s `-w '%{size_download}'`, or `wc -c` on the file you saved.

In [ ]:
%%bash
# write your command here


How big is it? Hyperschedule's page downloads this whole list **once** when it opens. So when you type in the search box, which machine does the searching? When would this design be a bad idea?

**Your answer:**



### 1.6 Two hostnames, two companies

Ask who serves `https://hyperschedule.io` (its `server:` header). Then ask the same about the API, `$B/v4/term/current`.

In [ ]:
%%bash
# write your command here


Who serves each one? Which hostname delivers the app's *files* and which delivers its *data*?

**Your answer:**



## Make it fail: four status codes

Get Hyperschedule's API to return each of **200, 404, 401, and 400**. Here are its routes, from [the source](https://github.com/hyperschedule/hyperschedule/tree/d41a5e52e72398ba504ceddb5d179a9ff85fe601/backend/src/routes/v4):

| Route | What it's for |
|---|---|
| `GET /v4/term/current` | the current term |
| `GET /v4/sections/{term}` | every section in one term, e.g. `FA2026` |
| `GET /v4/user/` | *your* account and saved schedules |
| `GET /v4/calendar/{userId}/{scheduleId}` | a calendar feed for one saved schedule |

### 1.7 Four codes

Write one `curl` line per code. Use `-w '\n[%{http_code}]\n'` so you see the body **and** the code. The 200 is the easy one: it's your request from 1.1.

In [ ]:
%%bash
# write your command here


Which URL gave each code? Read the **400** body: what is it complaining about? Find the line that sent it in [`calendar.ts`](https://github.com/hyperschedule/hyperschedule/blob/d41a5e52e72398ba504ceddb5d179a9ff85fe601/backend/src/routes/v4/calendar.ts#L36). For the 401: what would your browser have sent that `curl` didn't?

**Your answer:**



**✅ Checkpoint.** Compare your Part 1 answers with the key on the board. Then go on to Part 2.

---
# Part 2 — Wordle

**Your target:** <https://www.nytimes.com/games/wordle/index.html>

This time, nobody gives you the URLs. You have the same tools as in Part 1, plus your browser's devtools. The questions tell you *what* to find, not *how*. Hints are folded up under each question; try without them first.

Do **2.1–2.5**. If you finish early, there's more at the end.

> **Spoiler warning.** Wordle's page asks for today's puzzle. In the Network tab, **don't click on today's puzzle request** unless you've already played. In this notebook, use **past dates only**.

### 2.1 Find the puzzle

Open Wordle with devtools → **Network** open, then press **Play**. The page makes a lot of requests, and most of them are ads and analytics. How many requests did the page make in total? (The count is in the Network tab's bottom bar.) Find the **one** that delivers the puzzle.

Then fetch a **past** day's puzzle from this notebook and pretty-print it.

<details><summary>Hint 1</summary>

Set the Network filter to **Fetch/XHR**, which hides images, scripts, and styles. Then type <code>wordle</code> in the filter box.
</details>

<details><summary>Hint 2</summary>

Look for a <code>.json</code> URL with today's date in it. Keep everything up to the date, and swap the date for an old one, like <code>2026-09-20.json</code>.
</details>

In [47]:
%%bash
# write your command here

curl -sS -D - -o /dev/null https://www.nytimes.com/games/wordle/index.html | grep -i '^server'

server: envoy


What's the URL pattern? What fields does a puzzle have?

**Your answer:**



### 2.2 Who grades your guess?

Clear the Network tab (🚫). Type a guess and press **Enter**. Watch the Network tab.

How many requests did your guess cause? The tiles changed color anyway. So **which machine** decided they were green or yellow, and what did that machine need to know? Point to the evidence in your output from 2.1.

This is the video's question: *which machine runs this code?*

**Your answer:**



### 2.3 Where's your game?

Open devtools → **Application** → **Local Storage** → `https://www.nytimes.com`. Find the entry that holds your board. *(If you've played today, don't show your screen around.)*

The page **also** asked the NYT's server for your game state. Find that request in the Network tab (filter for `state`), copy its URL, and `curl` it here with the status code.

<details><summary>Hint</summary>

The URL contains <code>svc/games/state</code>. Use <code>curl -s -w '\n[%{http_code}]\n' "URL"</code>, and keep the double quotes, because the URL has a <code>?</code> in it.
</details>

In [ ]:
%%bash
# write your command here


What's in Local Storage? What did the state request return, and why? Which one survives switching laptops? Which part of Hyperschedule does this remind you of?

**Your answer:**



### 2.4 Make it fail — twice, differently

Get **two 404s** from the puzzle URL whose **bodies look completely different**. What does the difference tell you about how many programs sit between you and the puzzle?

<details><summary>Hint</summary>

Try a "date" that isn't a date at all. Then try a real date far in the future.
</details>

In [ ]:
%%bash
# write your command here


**Your answer:**



### 2.5 Fill in the diagram

Same boxes as the video, two apps. The Hyperschedule column is filled in from the video. Fill in Wordle's from what **you** found, and mark each box **[O]** observed (a command or devtools showed it) or **[G]** guessed (inferred).

| Layer | Hyperschedule (from the video) | Wordle (you) | [O] / [G] |
|---|---|---|---|
| The pixels | React components + CSS | | |
| Logic in the browser | search; guest schedule in Local Storage | | |
| The network | `hyperschedule.io` → GitHub Pages; `banana.hyperschedule.io` → Cloudflare | | |
| The API server | tinyhttp: `/v4/sections`, `/v4/user/` | | |
| The database | MongoDB (we read the source) | | |
| Below the bottom | the colleges' registrar, polled every 60 s | | |

**Last question:** What would it take to make cheating on Wordle impossible? Which boxes would have to change? Why do you think the NYT hasn't bothered?

*Your answer:*

---
## If you finish early

Four more, in any order. **A** and **B** go deeper into Wordle's data; **C** and **D** practice headers and loops.

### A. Is `id` the puzzle number?

Fetch a few **consecutive** past days and print `print_date`, `id`, and `days_since_launch` for each one.

<details><summary>Hint</summary>

<code>jq -c '{print_date, id, days_since_launch}'</code> prints one compact line per puzzle. Put three <code>curl</code> lines in the cell, or write a <code>for</code> loop.
</details>

In [ ]:
%%bash
# write your command here


Which field counts puzzles? What do you think `id` is? Is that **observed** or **guessed**?

**Your answer:**



### B. Wordle #1

Use `days_since_launch` to work out the date of the **first** Wordle, then fetch it. `date` can do date arithmetic: `date -d '2026-09-20 - 10 days' +%F`.

Compare its fields with a recent puzzle's. What's missing, and what do you think happened?

In [ ]:
%%bash
# write your command here


**Your answer:**



### C. Who answers?

Compare the headers of the puzzle JSON and the game page (`server`, `cache-control`, and anything that starts with `x-`). Is the page cached? Is the puzzle? Hyperschedule used two hostnames for files and data. Does Wordle?

### D. A week in one command

Write a `for` loop that prints the puzzle number and editor for each of the last seven days, **not including today**. `date -d "-3 days" +%F` gives the date three days ago.

In [ ]:
%%bash
# write your command here


**Your answer:**

